In [1]:
# Import the libraries and connect to the ENTSO-E API with the token
import pandas as pd
from entsoe import EntsoePandasClient

token = open("../entsoe_token.txt").read().strip()
client = EntsoePandasClient(api_key=token)

In [2]:
# Day-ahead prices for yesterday in four bidding zones
start = pd.Timestamp("2026-10-06", tz="Europe/Brussels")
end = pd.Timestamp("2026-10-07", tz="Europe/Brussels")

for zone in ["PT", "ES", "FR", "DE_LU"]:
    zone_prices = client.query_day_ahead_prices(zone, start=start, end=end)
    print(zone, len(zone_prices), "values, average", round(zone_prices.mean(), 1), "€/MWh")

PT 97 values, average 207.0 €/MWh
ES 97 values, average 202.3 €/MWh
FR 97 values, average 201.5 €/MWh
DE_LU 97 values, average 201.8 €/MWh


In [3]:
# Look at the Portuguese prices: time zone and 15-minute periods
pt_prices = client.query_day_ahead_prices("PT", start=start, end=end)
pt_prices.head(8)

2026-10-05 23:00:00+01:00    208.31
2026-10-05 23:15:00+01:00    207.31
2026-10-05 23:30:00+01:00    205.01
2026-10-05 23:45:00+01:00    203.00
2026-10-06 00:00:00+01:00    200.50
2026-10-06 00:15:00+01:00    200.31
2026-10-06 00:30:00+01:00    200.31
2026-10-06 00:45:00+01:00    199.02
dtype: float64

In [4]:
# Turn the 15-minute prices into hourly averages
pt_hourly = pt_prices.resample("h").mean()
pt_hourly.head(5)

2026-10-05 23:00:00+01:00    205.9075
2026-10-06 00:00:00+01:00    200.0350
2026-10-06 01:00:00+01:00    195.4100
2026-10-06 02:00:00+01:00    190.7675
2026-10-06 03:00:00+01:00    189.6000
Freq: h, dtype: float64

In [5]:
# Compare with the OMIE prices we already have for the same day
ours = pd.read_csv("../data/prices_hourly.csv")
ours_day = ours[ours["date"] == "2026-10-06"].set_index("hour")["price_pt"]
entsoe_day = pt_hourly.tz_convert("Europe/Lisbon")
entsoe_day = entsoe_day[entsoe_day.index.strftime("%Y-%m-%d") == "2026-10-06"]
comparison = pd.DataFrame({"omie": ours_day.values, "entsoe": entsoe_day.round(2).values})
comparison.head(10)

,omie,entsoe
0,200.04,200.04
1,195.41,195.41
2,190.77,190.77
3,189.60,189.60
4,189.79,189.79
5,208.36,208.36
6,236.58,236.58
7,245.09,245.09
8,243.11,243.11
9,202.12,202.12


In [6]:
# Wind, solar and consumption in Portugal for the same day
generation = client.query_generation("PT", start=start, end=end)
print(generation.columns.tolist())
load = client.query_load("PT", start=start, end=end)
load.head()

[('Fossil Gas', 'Actual Consumption'), ('Fossil Hard coal', 'Actual Consumption'), ('Hydro Pumped Storage', 'Actual Consumption'), ('Hydro Run-of-river and pondage', 'Actual Consumption'), ('Hydro Water Reservoir', 'Actual Consumption'), ('Biomass', 'Actual Aggregated'), ('Fossil Gas', 'Actual Aggregated'), ('Fossil Hard coal', 'Actual Aggregated'), ('Hydro Pumped Storage', 'Actual Aggregated'), ('Hydro Run-of-river and pondage', 'Actual Aggregated'), ('Hydro Water Reservoir', 'Actual Aggregated'), ('Solar', 'Actual Aggregated'), ('Wind Offshore', 'Actual Aggregated'), ('Wind Onshore', 'Actual Aggregated'), ('Other', 'Actual Aggregated')]


,Actual Load
2026-10-05 23:00:00+01:00,5230.9
2026-10-06 00:00:00+01:00,4914.2
2026-10-06 01:00:00+01:00,4685.3
2026-10-06 02:00:00+01:00,4514.3
2026-10-06 03:00:00+01:00,4430.8
